# From genomic coordinate to protein impact — KRAS G12D

This notebook walks a single point mutation through the SeqMat pipeline end-to-end:

1. Resolve `chr12:25,245,350` to the gene that contains it.
2. Inspect the gene model (transcripts, splice sites, primary).
3. Build the mature mRNA and translate the protein.
4. Apply the G12D substitution **on the genomic coordinate** and re-translate.
5. Diff the wild-type and mutant proteins.

Everything below is real — pulled live from a `genes.db` built off Ensembl 111 + the hg38 FASTA. No mock data, no detours.

> **Why this mutation?** *KRAS* G12D is one of the most common driver mutations in pancreatic, colorectal, and lung adenocarcinomas. It locks KRAS in its GTP-bound, signaling-active state. The single-letter change — Gly → Asp at protein position 12 — is what every oncologist's slide deck means by *"a KRAS-mutant tumor."*

In [1]:
from seqmat import Gene, SeqMat

## 1. Coordinate → gene

`chr12:25,245,350` is the position of the G12 codon on the reference. `Gene.from_position` does an interval-overlap lookup against the pre-built per-chromosome index — microseconds, no full table scan.

Chromosome strings can be `"12"`, `"chr12"`, or `"Chr12"` — all normalized.

In [2]:
hits = Gene.from_position("chr12", 25_245_350)
hits

[Gene(KRAS)]

In [3]:
kras = hits[0]
print(kras)
print("Strand:", "reverse" if kras.rev else "forward")

Gene: KRAS, ID: ENSG00000133703, Chr: 12, Transcripts: 14
Strand: reverse


*KRAS lives on chromosome 12, encoded on the reverse strand.* The reverse-strand bookkeeping is handled by SeqMat throughout — the rest of this notebook never has to think about it.

## 2. Gene structure

Every gene exposes its annotated transcripts and aggregated splice sites.

In [4]:
print(f"Transcripts: {len(kras)}")
for tx in list(kras)[:5]:
    print(f"  {tx.transcript_id}  biotype={tx.transcript_biotype}  protein_coding={tx.protein_coding}")

Transcripts: 14
  ENST00000256078  biotype=protein_coding  protein_coding=True
  ENST00000311936  biotype=protein_coding  protein_coding=True
  ENST00000556131  biotype=protein_coding  protein_coding=True
  ENST00000557334  biotype=protein_coding  protein_coding=True
  ENST00000685328  biotype=protein_coding  protein_coding=True


In [5]:
acceptors, donors = kras.splice_sites()
print(f"{len(acceptors)} unique acceptors, {len(donors)} unique donors across all transcripts")
print("Most-used acceptors:", acceptors.most_common(3))
print("Most-used donors:   ", donors.most_common(3))

10 unique acceptors, 10 unique donors across all transcripts
Most-used acceptors: [(25209911, 12), (25245395, 11), (25225773, 10)]
Most-used donors:    [(25245274, 11), (25225614, 11), (25250751, 9)]


## 3. Wild-type mRNA and protein

`gene.transcript()` returns the canonical transcript (Ensembl-flagged primary, else the first protein-coding). `generate_mature_mrna()` assembles the spliced mRNA from exon coordinates; `generate_protein()` translates from TIS to TTS.

In [6]:
tx = kras.transcript()
print("Primary transcript:", tx.transcript_id)
print("TIS / TTS (genomic):", tx.TIS, "/", tx.TTS)

tx.generate_mature_mrna()
tx.generate_protein()

wt_protein = tx.protein
print(f"\nWT protein ({len(wt_protein)} aa):")
print(wt_protein)

Primary transcript: ENST00000311936
TIS / TTS (genomic): 25245384 / 25209798

WT protein (188 aa):
MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQEEYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHHYREQIKRVKDSEDVPMVLVGNKCDLPSRTVDTKQAQDLARSYGIPFIETSAKTRQGVDDAFYTLVREIRKHKEKMSKDGKKKKKKSKTKCVIM


The protein starts with `MTEYKLVVVGAGGVGKSALT...` — `M T E Y K L V V V G A G G V G...` — the GAGGVG fingerprint of the P-loop. Position 12 is the second G in `GAGGVG`.

## 4. Apply G12D — at the genomic coordinate

G12D on the reverse-strand KRAS gene corresponds to `chr12:25,245,350 C>T` on the forward strand of the reference. We mutate the pre-mRNA at that coordinate; SeqMat's mutation log tracks it.

Re-running `generate_mature_mrna` → `generate_protein` re-splices and re-translates from the mutated genomic sequence.

In [7]:
tx_mut = kras.transcript()
tx_mut.generate_pre_mrna()
tx_mut.pre_mrna.apply_mutations([(25_245_350, "C", "T")])
tx_mut.pre_mrna.mutations

[{'pos': 25245350, 'ref': 'C', 'alt': 'T', 'type': 'snp'}]

In [8]:
tx_mut.generate_mature_mrna()
tx_mut.generate_protein()
mut_protein = tx_mut.protein
print(f"Mutant protein ({len(mut_protein)} aa):")
print(mut_protein)

Mutant protein (188 aa):
MTEYKLVVVGADGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQEEYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHHYREQIKRVKDSEDVPMVLVGNKCDLPSRTVDTKQAQDLARSYGIPFIETSAKTRQGVDDAFYTLVREIRKHKEKMSKDGKKKKKKSKTKCVIM


## 5. Diff

One single-nucleotide change at one genomic coordinate produces exactly one amino-acid change — at position 12, as expected.

In [9]:
diffs = [(i + 1, wt, mt) for i, (wt, mt) in enumerate(zip(wt_protein, mut_protein)) if wt != mt]
print(f"Number of amino-acid differences: {len(diffs)}")
for pos, wt_aa, mut_aa in diffs:
    print(f"  position {pos}: {wt_aa} -> {mut_aa}")

Number of amino-acid differences: 1
  position 12: G -> D


In [10]:
# Side-by-side around the mutation site
window = slice(0, 25)
print("WT :", wt_protein[window])
print("MUT:", mut_protein[window])
print("     " + "".join("^" if a != b else " " for a, b in zip(wt_protein[window], mut_protein[window])))

WT : MTEYKLVVVGAGGVGKSALTIQLIQ
MUT: MTEYKLVVVGADGVGKSALTIQLIQ
                ^             


---

## Recap

Five short cells took us from a chromosome coordinate to a mechanistic protein-level readout of the most-studied oncogenic point mutation in cancer. The library handled:

- **Interval lookup** — `Gene.from_position` against a per-chromosome NumPy index (~2.5 µs).
- **Reverse-strand bookkeeping** — KRAS is on the minus strand; the user never has to think about it.
- **Splicing** — `generate_mature_mrna()` stitches exons directly without re-scanning the pre-mRNA.
- **Mutation tracking** — every change recorded with type / ref / alt for later auditing or alignment.
- **Translation** — start-to-stop, no third-party translator needed.

Substitute any other gene + variant and the same five cells produce the same kind of answer.